In [ ]:
# Figure 2E–F: tracheole volume fraction by neuropil
# Input: 110 segmented tracheoles (7 peripheral tracheoles excluded due to incomplete segmentation).
# Method: MIP2 voxels clipped to tracheole bounding boxes, point-in-mesh test, neuropil mesh volumes.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import trimesh
from matplotlib.collections import PolyCollection
from matplotlib.colors import Normalize, PowerNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

from fafbseg import flywire
import navis

# Input folder contains tracheole_root_bboxes.csv and root_voxels/*.npz.
data_dir = Path('data')
voxel_dir = data_dir / 'tracheole_voxel_volume'
mesh_dir = data_dir / 'neuropil_mesh'
out_dir = Path('fig2_outputs')
mesh_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)
point_chunk = 500_000
excluded_regions = {'LA_L', 'LA_R'}

pairs = pd.read_csv(voxel_dir / 'tracheole_root_bboxes.csv', dtype={'flywire_root_id': 'string'})

def exact_root_id(value) -> str:
    if value is None or pd.isna(value):
        return ''
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    text = str(value).strip()
    if text.endswith('.0'):
        text = text[:-2]
    digits = ''.join((ch for ch in text if ch.isdigit()))
    if 15 <= len(digits) <= 20:
        return digits
    raise ValueError(f'Could not parse FlyWire root ID from {value!r}')

def bbox_columns(df: pd.DataFrame):
    required = ['bbox_min_x_nm', 'bbox_min_y_nm', 'bbox_min_z_nm', 'bbox_max_x_nm', 'bbox_max_y_nm', 'bbox_max_z_nm']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f'Missing bbox columns: {missing}')
    return required

def row_bbox(row):
    lo = np.array([row.bbox_min_x_nm, row.bbox_min_y_nm, row.bbox_min_z_nm], dtype=np.float64)
    hi = np.array([row.bbox_max_x_nm, row.bbox_max_y_nm, row.bbox_max_z_nm], dtype=np.float64)
    return (lo, hi)

bbox_columns(pairs)
pairs['skeleton_id'] = pd.to_numeric(pairs['skeleton_id'], errors='raise').astype(np.int64)
pairs['flywire_root_id'] = pairs['flywire_root_id'].map(exact_root_id).astype('string')
assert pairs['skeleton_id'].nunique() == 110 and pairs['flywire_root_id'].str.len().gt(0).all()

# Load cached MIP2 segmentation voxels for each FlyWire root.
def root_cache_path(root_id, group):
    lo = group[['bbox_min_x_nm', 'bbox_min_y_nm', 'bbox_min_z_nm']].to_numpy(float).min(axis=0)
    hi = group[['bbox_max_x_nm', 'bbox_max_y_nm', 'bbox_max_z_nm']].to_numpy(float).max(axis=0)
    path = voxel_dir / 'root_voxels' / f'{root_id}_mip2_voxels.npz'
    with np.load(path, allow_pickle=False) as z:
        files = set(z.files)
        if 'flywire_root_id' in files and exact_root_id(np.asarray(z['flywire_root_id']).reshape(-1)[0]) != root_id:
            raise ValueError(f'Root ID mismatch in {path.name}')
        if 'mip' in files and int(np.asarray(z['mip']).reshape(-1)[0]) != 2:
            raise ValueError(f'Expected MIP2 voxel cache: {path.name}')
        if {'union_bbox_min_nm', 'union_bbox_max_nm'} <= files:
            covers = np.all(np.asarray(z['union_bbox_min_nm']) <= lo + 1) and np.all(np.asarray(z['union_bbox_max_nm']) >= hi - 1)
        elif 'selection_bounds_base' in files:
            bounds = np.asarray(z['selection_bounds_base'], dtype=np.int64)
            if bounds.shape == (2, 3): bounds = bounds.T
            covers = (bounds.shape == (3, 2) and
                      np.all(bounds[:, 0] <= np.floor(lo / [4., 4., 40.])) and
                      np.all(bounds[:, 1] >= np.ceil(hi / [4., 4., 40.])))
        else:
            covers = False
    if not covers:
        raise ValueError(f'MIP2 voxel cache does not cover the tracheole bounding boxes: {root_id}')
    return path

# Neuropil meshes are stored in FlyWire nanometre coordinates.
def load_neuropil(name):
    path = mesh_dir / f'{name}.npz'
    if path.exists():
        with np.load(path) as z:
            vertices, faces = np.asarray(z['vertices'], float), np.asarray(z['faces'], np.int64)
        volume = navis.Volume(vertices, faces, name=name)
    else:
        volume = flywire.get_neuropil_volumes(name)
        vertices, faces = np.asarray(volume.vertices, float), np.asarray(volume.faces, np.int64)
        np.savez_compressed(path, vertices=vertices, faces=faces)
    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    return dict(volume=volume, vertices=vertices, faces=faces,
                bounds=np.asarray(mesh.bounds, float), volume_um3=abs(float(mesh.volume)) / 1e9)

if list(mesh_dir.glob('*.npz')):
    region_names = sorted(p.stem for p in mesh_dir.glob('*.npz') if p.stem not in excluded_regions)
else:
    flywire.set_default_dataset('public')
    region_names = sorted(set(map(str, np.asarray(flywire.get_neuropil_volumes(None)).ravel())) - excluded_regions)
neuropils = {name: load_neuropil(name) for name in region_names}

try:
    import ncollpyde
    colliders = {name: ncollpyde.Volume(d['vertices'], d['faces'], n_rays=3) for name, d in neuropils.items()}
    def points_inside(name, xyz):
        return np.concatenate([np.asarray(colliders[name].contains(xyz[i:i + point_chunk]), bool)
                               for i in range(0, len(xyz), point_chunk)]) if len(xyz) else np.zeros(0, bool)
except ImportError:
    def points_inside(name, xyz):
        return np.concatenate([np.asarray(navis.in_volume(xyz[i:i + point_chunk], neuropils[name]['volume']), bool)
                               for i in range(0, len(xyz), point_chunk)]) if len(xyz) else np.zeros(0, bool)

# Each root is clipped to the union of its reviewed tracheole boxes before neuropil assignment.
region_counts = dict.fromkeys(region_names, 0)
voxel_volumes = []
for root_i, (root_id, group) in enumerate(pairs.groupby('flywire_root_id', sort=True), 1):
    root_id = exact_root_id(root_id)
    with np.load(root_cache_path(root_id, group), allow_pickle=False) as z:
        voxels, resolution_nm = np.asarray(z['voxels'], np.int64), np.asarray(z['resolution_nm'], float)
    voxel_volumes.append(float(np.prod(resolution_nm)) / 1e9)
    centers = (voxels.astype(float) + 0.5) * resolution_nm
    reviewed = np.zeros(len(centers), bool)
    for row in group.itertuples(index=False):
        lo, hi = row_bbox(row)
        reviewed |= np.all(centers >= lo, axis=1) & np.all(centers <= hi, axis=1)
    xyz = centers[reviewed]
    for name in region_names:
        lower, upper = neuropils[name]['bounds']
        candidate = xyz[np.all(xyz >= lower, axis=1) & np.all(xyz <= upper, axis=1)]
        if len(candidate): region_counts[name] += int(points_inside(name, candidate).sum())

assert np.allclose(voxel_volumes, voxel_volumes[0]), 'Inconsistent voxel resolutions.'
voxel_volume_um3 = voxel_volumes[0]
result_df = pd.DataFrame([{'neuropil': name, 'trachea_voxel_count': region_counts[name],
                           'trachea_volume_um3': region_counts[name] * voxel_volume_um3,
                           'neuropil_volume_um3': neuropils[name]['volume_um3'],
                           'tracheation_percent': 100 * region_counts[name] * voxel_volume_um3 / neuropils[name]['volume_um3']}
                          for name in region_names])
result_df = result_df.sort_values('tracheation_percent', ascending=False).reset_index(drop=True)
result_df.to_csv(out_dir / 'tracheation_by_neuropil.csv', index=False)
print(f"76 neuropils, median tracheole volume fraction: {result_df['tracheation_percent'].median():.4f}%")

# Figure 2E: volume fraction in all 76 neuropil subregions.
plt.rcParams.update({'font.family': 'Arial', 'font.size': 9, 'svg.fonttype': 'none'})
vals = result_df['tracheation_percent'].to_numpy(float)
base_cmap = plt.get_cmap('GnBu')
cmap = LinearSegmentedColormap.from_list('GnBu_truncated', base_cmap(np.linspace(0.16, 1, 256)))
bar_norm = Normalize(vmin=0, vmax=0.6)
fig, ax = plt.subplots(figsize=(7.8, 0.28 * len(result_df) + 1.2))
y = np.arange(len(result_df))
ax.barh(y, vals, color=cmap(bar_norm(vals)), edgecolor='none', height=0.72)
ax.set_yticks(y, result_df['neuropil']); ax.invert_yaxis()
ax.set_xlabel('Tracheole volume fraction (%)'); ax.set_xlim(0, max(vals) + .06)
for yy, value in zip(y, vals): ax.text(value + .004, yy, f'{value:.3f}%', va='center', fontsize=7)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); fig.savefig(out_dir / 'fig2e.svg', bbox_inches='tight'); plt.show()

# Figure 2F: frontal and dorsal orthographic views using the triangle/depth projection.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
mesh_norm = PowerNorm(gamma=0.5, vmin=0, vmax=0.55, clip=True)
mesh_cmap = LinearSegmentedColormap.from_list('GnBu_mesh', base_cmap(np.linspace(0.14, 1, 256)))
val_by_region = result_df.set_index('neuropil')['tracheation_percent'].to_dict()
all_vertices = np.vstack([neuropils[name]['vertices'] for name in region_names])
low, high = all_vertices.min(axis=0), all_vertices.max(axis=0)
right_minus_left = [np.median(neuropils[n[:-2] + '_R']['vertices'][:, 0]) - np.median(neuropils[n]['vertices'][:, 0])
                    for n in region_names if n.endswith('_L') and n[:-2] + '_R' in neuropils]
right_positive_x = np.median(right_minus_left) > 0
for ax, (name, horizontal, vertical, depth, invert_vertical) in zip(axes,
        [('Frontal', 0, 1, 2, False), ('Dorsal', 0, 2, 1, True)]):
    triangles, depths, colors = [], [], []
    for region in region_names:
        vertices, faces = neuropils[region]['vertices'], neuropils[region]['faces']
        if len(faces) > 6000: faces = faces[np.linspace(0, len(faces) - 1, 6000, dtype=int)]
        tri = vertices[faces]
        triangles.append(tri[:, :, [horizontal, vertical]])
        depths.append(tri[:, :, depth].mean(axis=1))
        colors.append(np.tile(mesh_cmap(mesh_norm(val_by_region[region])), (len(tri), 1)))
    triangles, depths, colors = np.concatenate(triangles), np.concatenate(depths), np.concatenate(colors)
    order = np.argsort(-depths)
    ax.add_collection(PolyCollection(triangles[order], facecolors=colors[order], edgecolors='none', alpha=0.97, rasterized=True))
    dx, dy = high[horizontal] - low[horizontal], high[vertical] - low[vertical]
    ax.set_xlim(low[horizontal] - .025*dx, high[horizontal] + .025*dx)
    ax.set_ylim(low[vertical] - .025*dy, high[vertical] + .025*dy)
    if horizontal == 0 and not right_positive_x: ax.invert_xaxis()
    if invert_vertical: ax.invert_yaxis()
    ax.set_aspect('equal'); ax.set_axis_off(); ax.set_title(name)
fig.colorbar(ScalarMappable(norm=mesh_norm, cmap=mesh_cmap), ax=axes, fraction=.025, pad=.02, label='Tracheole volume fraction (%)')
fig.savefig(out_dir / 'fig2f.svg', bbox_inches='tight'); plt.show()

pd.DataFrame([{'figure': '2E–F', 'n_neuropils': len(result_df),
               'median_fraction_percent': float(np.median(vals)),
               'q25_percent': float(np.quantile(vals, .25)),
               'q75_percent': float(np.quantile(vals, .75))}]).to_csv(out_dir / 'fig2_stats.csv', index=False)